# 01 — Validation XML — XSD, DTD, RelaxNG

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre les différents types de schémas XML (DTD, XSD, RelaxNG) ;
- valider un document XML contre un schéma XSD avec `lxml` ;
- générer des messages d’erreur compréhensibles ;
- utiliser le package `xmlschema` pour valider et convertir en dict/JSON ;
- écrire un XSD simple (elements, types, cardinalité) ;
- gérer les namespaces dans les schémas.

## Prérequis — ce que vous devez maîtriser

À ce stade de la formation, vous savez :

- bases XML (parsing avec `lxml` ou `xml.etree`) ;
- fonctions Python typées.

Ce que ce notebook **ne couvrira pas** :

- XSLT (voir `03_Xpath_et_xslt.ipynb`) ;
- Schematron (validation par règles) ;
- streaming XML (voir `02_Sax_et_iterparse.ipynb`).

## Plan

1. Installation
2. DTD — la plus ancienne
3. XSD — écrire un schéma
4. Valider avec `lxml.etree.XMLSchema`
5. Messages d’erreur
6. Le package `xmlschema`
7. Conversion XML → dict / JSON
8. Types restreints (enumeration, pattern)
9. Namespaces dans les schémas
10. RelaxNG
11. Synthèse
12. Exercices
13. Ressources externes

### Exemple avancé : XSD avec types complexes imbriqués

Les schémas XSD réels contiennent souvent des types complexes imbriqués,
des attributs, et des restrictions sur les types simples.

In [ ]:
xsd_complex = '''
<xs:schema xmlns:xs="http://www.w3.org/2001/XMLSchema">
  <xs:element name="catalogue">
    <xs:complexType>
      <xs:sequence>
        <xs:element name="livre" maxOccurs="unbounded">
          <xs:complexType>
            <xs:sequence>
              <xs:element name="titre" type="xs:string"/>
              <xs:element name="auteur" type="xs:string"/>
              <xs:element name="prix">
                <xs:simpleType>
                  <xs:restriction base="xs:decimal">
                    <xs:minInclusive value="0"/>
                    <xs:maxInclusive value="999.99"/>
                  </xs:restriction>
                </xs:simpleType>
              </xs:element>
            </xs:sequence>
            <xs:attribute name="isbn" type="xs:string" use="required"/>
            <xs:attribute name="langue" type="xs:string" default="fr"/>
          </xs:complexType>
        </xs:element>
      </xs:sequence>
    </xs:complexType>
  </xs:element>
</xs:schema>
'''
print(xsd_complex)

### Exercice : validation de données produit

1. Écrivez un schéma XSD pour un catalogue de produits avec :
   - Un produit a un `nom` (string), `prix` (decimal > 0), `stock` (integer >= 0)
   - Un attribut `categorie` obligatoire
2. Créez un document XML valide et un invalide
3. Validez les deux avec `lxml.etree.XMLSchema`

In [ ]:
from lxml import etree

# Votre schéma XSD
xsd_produit = '''
<xs:schema xmlns:xs="http://www.w3.org/2001/XMLSchema">
  <xs:element name="catalogue">
    <xs:complexType>
      <xs:sequence>
        <xs:element name="produit" maxOccurs="unbounded">
          <xs:complexType>
            <xs:sequence>
              <xs:element name="nom" type="xs:string"/>
              <xs:element name="prix">
                <xs:simpleType>
                  <xs:restriction base="xs:decimal">
                    <xs:minExclusive value="0"/>
                  </xs:restriction>
                </xs:simpleType>
              </xs:element>
              <xs:element name="stock" type="xs:nonNegativeInteger"/>
            </xs:sequence>
            <xs:attribute name="categorie" type="xs:string" use="required"/>
          </xs:complexType>
        </xs:element>
      </xs:sequence>
    </xs:complexType>
  </xs:element>
</xs:schema>
'''

schema = etree.XMLSchema(etree.fromstring(xsd_produit.encode()))

# Document valide
xml_ok = '<catalogue><produit categorie="info"><nom>Clavier</nom><prix>49.99</prix><stock>150</stock></produit></catalogue>'
print('Valide :', schema.validate(etree.fromstring(xml_ok.encode())))

# Document invalide (prix négatif)
xml_ko = '<catalogue><produit categorie="info"><nom>Clavier</nom><prix>-5</prix><stock>10</stock></produit></catalogue>'
print('Invalide :', schema.validate(etree.fromstring(xml_ko.encode())))
print('Erreur :', schema.error_log)

### Exercice : RelaxNG vs XSD

RelaxNG est un format de schéma alternatif, plus simple que XSD.
Comparez la validation d'un même document avec les deux formats.

lxml supporte RelaxNG via `etree.RelaxNG`.

In [ ]:
# Schéma RelaxNG compact (XML syntax)
rng = '''
<element name="personne" xmlns="http://relaxng.org/ns/structure/1.0">
  <element name="nom"><text/></element>
  <element name="age"><data type="integer" datatypeLibrary="http://www.w3.org/2001/XMLSchema-datatypes"/></element>
</element>
'''

relaxng = etree.RelaxNG(etree.fromstring(rng.encode()))
doc = etree.fromstring(b'<personne><nom>Alice</nom><age>30</age></personne>')
print('RelaxNG valide :', relaxng.validate(doc))

---

## 1. Installation

In [ ]:
# pip install lxml xmlschema
from lxml import etree
import xmlschema
import json
print(f"lxml {etree.LXML_VERSION}")

---

## 2. DTD — la plus ancienne

DTD (Document Type Definition) est historique mais limitée : pas de types, pas de namespaces.

In [ ]:
dtd_text = '<!ELEMENT personne (nom, age)>\n<!ELEMENT nom (#PCDATA)>\n<!ELEMENT age (#PCDATA)>'
dtd = etree.DTD(io=None)  # on peut charger depuis un fichier
print('DTD : syntaxe très différente de XML, limitée')

In [ ]:
# Validation DTD avec lxml
from io import StringIO
dtd = etree.DTD(StringIO('<!ELEMENT p (nom, age)>\n<!ELEMENT nom (#PCDATA)>\n<!ELEMENT age (#PCDATA)>'))
doc = etree.fromstring('<p><nom>Alice</nom><age>30</age></p>')
print(f"Valide: {dtd.validate(doc)}")

---

## 3. XSD — écrire un schéma

XSD (XML Schema Definition) est le standard W3C. Beaucoup plus puissant que DTD.

In [ ]:
XSD = '''<?xml version='1.0' encoding='UTF-8'?>
<xs:schema xmlns:xs='http://www.w3.org/2001/XMLSchema'>
  <xs:element name='personne'>
    <xs:complexType>
      <xs:sequence>
        <xs:element name='nom' type='xs:string'/>
        <xs:element name='age' type='xs:integer'/>
        <xs:element name='email' type='xs:string' minOccurs='0'/>
      </xs:sequence>
    </xs:complexType>
  </xs:element>
</xs:schema>'''
print(f"XSD: {len(XSD)} caractères")

### Concepts clés XSD

| Concept | Rôle |
|---|---|
| `xs:element` | Définit un élément |
| `xs:complexType` | Type avec sous-éléments |
| `xs:simpleType` | Type sans sous-éléments (restriction) |
| `xs:sequence` | Ordre imposé |
| `xs:choice` | Un parmi plusieurs |
| `xs:all` | Tout, dans n’importe quel ordre |
| `minOccurs` / `maxOccurs` | Cardinalité |

---

## 4. Valider avec `lxml.etree.XMLSchema`

In [ ]:
schema = etree.XMLSchema(etree.fromstring(XSD.encode()))

xml_ok = '<personne><nom>Alice</nom><age>30</age></personne>'
doc = etree.fromstring(xml_ok)
print(f"Valide: {schema.validate(doc)}")

In [ ]:
xml_bad = '<personne><nom>Alice</nom><age>vingt</age></personne>'
doc_bad = etree.fromstring(xml_bad)
print(f"Valide: {schema.validate(doc_bad)}")

---

## 5. Messages d’erreur

In [ ]:
for err in schema.error_log:
    print(f"Ligne {err.line}: {err.message}")

In [ ]:
# assertValid lève une exception
try:
    schema.assertValid(doc_bad)
except etree.DocumentInvalid as e:
    print(f"Erreur: {e}")

---

## 6. Le package `xmlschema`

Plus pythonique que `lxml`, avec conversion automatique en dict/JSON.

In [ ]:
xs = xmlschema.XMLSchema(XSD)
print(f"Valide: {xs.is_valid('<personne><nom>Bob</nom><age>25</age></personne>')}")

In [ ]:
# Itérer sur les erreurs
for err in xs.iter_errors('<personne><age>10</age></personne>'):
    print(f"Erreur: {err.message[:80]}")

---

## 7. Conversion XML → dict / JSON

In [ ]:
d = xs.to_dict('<personne><nom>Alice</nom><age>30</age></personne>')
print(d)

In [ ]:
d = xs.to_dict('<personne><nom>Alice</nom><age>30</age><email>a@b.c</email></personne>')
print(json.dumps(d, indent=2, ensure_ascii=False))

---

## 8. Types restreints (enumeration, pattern)

In [ ]:
XSD_ENUM = '''<xs:schema xmlns:xs='http://www.w3.org/2001/XMLSchema'>
  <xs:element name='status'>
    <xs:simpleType>
      <xs:restriction base='xs:string'>
        <xs:enumeration value='active'/>
        <xs:enumeration value='pending'/>
        <xs:enumeration value='closed'/>
      </xs:restriction>
    </xs:simpleType>
  </xs:element>
</xs:schema>'''
xs_enum = xmlschema.XMLSchema(XSD_ENUM)
print(xs_enum.is_valid('<status>active</status>'))
print(xs_enum.is_valid('<status>unknown</status>'))

In [ ]:
# Pattern (regex)
XSD_PAT = '''<xs:schema xmlns:xs='http://www.w3.org/2001/XMLSchema'>
  <xs:element name='code'>
    <xs:simpleType>
      <xs:restriction base='xs:string'>
        <xs:pattern value='[A-Z]{3}-[0-9]{4}'/>
      </xs:restriction>
    </xs:simpleType>
  </xs:element>
</xs:schema>'''
xs_pat = xmlschema.XMLSchema(XSD_PAT)
print(xs_pat.is_valid('<code>ABC-1234</code>'))
print(xs_pat.is_valid('<code>abc-12</code>'))

---

## 9. Namespaces dans les schémas

In [ ]:
XSD_NS = '''<xs:schema xmlns:xs='http://www.w3.org/2001/XMLSchema'
  targetNamespace='http://example.com/ns'
  xmlns='http://example.com/ns'
  elementFormDefault='qualified'>
  <xs:element name='item' type='xs:string'/>
</xs:schema>'''
print('targetNamespace déclaré')

### Points clés

- `targetNamespace` : le namespace du schéma
- `elementFormDefault='qualified'` : les éléments doivent être qualifiés
- Les documents XML doivent déclarer le namespace correspondant

---

## 10. RelaxNG

RelaxNG est une alternative à XSD, plus simple et plus lisible.

In [ ]:
RNG = '''<element name='personne' xmlns='http://relaxng.org/ns/structure/1.0'>
  <element name='nom'><text/></element>
  <element name='age'><text/></element>
</element>'''
rng = etree.RelaxNG(etree.fromstring(RNG))
print(rng.validate(etree.fromstring('<personne><nom>A</nom><age>1</age></personne>')))

---

## Synthèse

| Opération | lxml | xmlschema |
|---|---|---|
| Charger XSD | `etree.XMLSchema(etree.fromstring(xsd))` | `XMLSchema(xsd)` |
| Valider (bool) | `schema.validate(doc)` | `xs.is_valid(xml)` |
| Valider (raise) | `schema.assertValid(doc)` | `xs.validate(xml)` |
| Erreurs | `schema.error_log` | `xs.iter_errors(xml)` |
| To dict | - | `xs.to_dict(xml)` |
| DTD | `etree.DTD(StringIO(dtd))` | - |
| RelaxNG | `etree.RelaxNG(tree)` | - |

### Règles à retenir

- **XSD** est le standard le plus utilisé (W3C).
- **`lxml`** est plus rapide ; **`xmlschema`** est plus pythonique.
- **Namespaces** via `targetNamespace`.
- **`minOccurs` / `maxOccurs`** pour la cardinalité.
- **DTD** est obsolète ; préférer XSD ou RelaxNG.

---

---

## Attributs dans XSD

In [ ]:
XSD_ATTR = '''<xs:schema xmlns:xs='http://www.w3.org/2001/XMLSchema'>
  <xs:element name='produit'>
    <xs:complexType>
      <xs:simpleContent>
        <xs:extension base='xs:string'>
          <xs:attribute name='prix' type='xs:decimal' use='required'/>
          <xs:attribute name='stock' type='xs:integer' use='optional'/>
        </xs:extension>
      </xs:simpleContent>
    </xs:complexType>
  </xs:element>
</xs:schema>'''
print('Attributs: use=required ou optional')

In [ ]:
xs_attr = xmlschema.XMLSchema(XSD_ATTR)
print(xs_attr.is_valid('<produit prix="9.99">Widget</produit>'))
print(xs_attr.is_valid('<produit>Widget</produit>'))  # manque prix = False

---

## Sequence vs Choice vs All

| Compositeur | Signification |
|---|---|
| `xs:sequence` | Tous les elements, dans l'ordre |
| `xs:choice` | Un seul element parmi les proposes |
| `xs:all` | Tous les elements, dans n'importe quel ordre |

In [ ]:
XSD_CHOICE = '''<xs:schema xmlns:xs='http://www.w3.org/2001/XMLSchema'>
  <xs:element name='contact'>
    <xs:complexType>
      <xs:choice>
        <xs:element name='email' type='xs:string'/>
        <xs:element name='telephone' type='xs:string'/>
      </xs:choice>
    </xs:complexType>
  </xs:element>
</xs:schema>'''
xs_ch = xmlschema.XMLSchema(XSD_CHOICE)
print(xs_ch.is_valid('<contact><email>a@b.c</email></contact>'))
print(xs_ch.is_valid('<contact><telephone>0123</telephone></contact>'))

---

## Types complexes reutilisables

In [ ]:
XSD_REUSE = '''<xs:schema xmlns:xs='http://www.w3.org/2001/XMLSchema'>
  <xs:complexType name='adresseType'>
    <xs:sequence>
      <xs:element name='rue' type='xs:string'/>
      <xs:element name='ville' type='xs:string'/>
      <xs:element name='code_postal' type='xs:string'/>
    </xs:sequence>
  </xs:complexType>
  <xs:element name='personne'>
    <xs:complexType>
      <xs:sequence>
        <xs:element name='nom' type='xs:string'/>
        <xs:element name='adresse' type='adresseType'/>
      </xs:sequence>
    </xs:complexType>
  </xs:element>
</xs:schema>'''
xs_r = xmlschema.XMLSchema(XSD_REUSE)
xml = '<personne><nom>Alice</nom><adresse><rue>1 rue</rue><ville>Paris</ville><code_postal>75001</code_postal></adresse></personne>'
print(xs_r.is_valid(xml))

In [ ]:
import json
print(json.dumps(xs_r.to_dict(xml), indent=2, ensure_ascii=False))

---

## Restrictions numeriques

In [ ]:
XSD_RANGE = '''<xs:schema xmlns:xs='http://www.w3.org/2001/XMLSchema'>
  <xs:element name='age'>
    <xs:simpleType>
      <xs:restriction base='xs:integer'>
        <xs:minInclusive value='0'/>
        <xs:maxInclusive value='150'/>
      </xs:restriction>
    </xs:simpleType>
  </xs:element>
</xs:schema>'''
xs_range = xmlschema.XMLSchema(XSD_RANGE)
for v in [0, 25, 150, 200, -1]:
    print(f"age={v}: {xs_range.is_valid(f'<age>{v}</age>')}")

---

## Listes et unions de types

In [ ]:
XSD_LIST = '''<xs:schema xmlns:xs='http://www.w3.org/2001/XMLSchema'>
  <xs:element name='scores'>
    <xs:simpleType>
      <xs:list itemType='xs:integer'/>
    </xs:simpleType>
  </xs:element>
</xs:schema>'''
import xmlschema
xs_list = xmlschema.XMLSchema(XSD_LIST)
print(xs_list.is_valid('<scores>10 20 30 40</scores>'))
print(xs_list.to_dict('<scores>10 20 30 40</scores>'))

---

## Validation en batch

In [ ]:
import xmlschema

XSD_SIMPLE = '''<xs:schema xmlns:xs='http://www.w3.org/2001/XMLSchema'>
  <xs:element name='n' type='xs:integer'/>
</xs:schema>'''
xs = xmlschema.XMLSchema(XSD_SIMPLE)

documents = ['<n>42</n>', '<n>abc</n>', '<n>0</n>', '<n>-5</n>']
for doc in documents:
    valid = xs.is_valid(doc)
    print(f"{doc:15s} -> {'OK' if valid else 'INVALIDE'}")

In [ ]:
# Lister toutes les erreurs d'un coup
for doc in documents:
    errors = list(xs.iter_errors(doc))
    if errors:
        for e in errors:
            print(f"  {doc}: {e.reason}")

## Exercices

### Exercice 1 *(facile)* — Validation simple

Validez `<personne><nom>Alice</nom><age>30</age></personne>` contre le XSD donné plus haut.

In [ ]:
from lxml import etree

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from lxml import etree

xsd = '''<xs:schema xmlns:xs='http://www.w3.org/2001/XMLSchema'>
<xs:element name='personne'><xs:complexType><xs:sequence>
<xs:element name='nom' type='xs:string'/>
<xs:element name='age' type='xs:integer'/>
</xs:sequence></xs:complexType></xs:element>
</xs:schema>'''
schema = etree.XMLSchema(etree.fromstring(xsd))
doc = etree.fromstring('<personne><nom>Alice</nom><age>30</age></personne>')
print(f"Valide: {schema.validate(doc)}")
```

</details>

### Exercice 2 *(facile)* — Document invalide

Écrivez un code qui attrape l’erreur quand `age` contient du texte au lieu d’un entier.

In [ ]:
from lxml import etree

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from lxml import etree

xsd = '''<xs:schema xmlns:xs='http://www.w3.org/2001/XMLSchema'>
<xs:element name='p'><xs:complexType><xs:sequence>
<xs:element name='age' type='xs:integer'/>
</xs:sequence></xs:complexType></xs:element>
</xs:schema>'''
schema = etree.XMLSchema(etree.fromstring(xsd))
doc = etree.fromstring('<p><age>vingt</age></p>')
try:
    schema.assertValid(doc)
except etree.DocumentInvalid as e:
    print(f"Erreur: {e}")
```

</details>

### Exercice 3 *(moyen)* — Liste de personnes

Écrivez un XSD pour un élément `personnes` contenant 1 à N éléments `personne` (nom, age). Validez un exemple.

In [ ]:
# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import xmlschema

xsd = '''<xs:schema xmlns:xs='http://www.w3.org/2001/XMLSchema'>
<xs:element name='personnes'>
  <xs:complexType><xs:sequence>
    <xs:element name='personne' minOccurs='1' maxOccurs='unbounded'>
      <xs:complexType><xs:sequence>
        <xs:element name='nom' type='xs:string'/>
        <xs:element name='age' type='xs:integer'/>
      </xs:sequence></xs:complexType>
    </xs:element>
  </xs:sequence></xs:complexType>
</xs:element>
</xs:schema>'''
xs = xmlschema.XMLSchema(xsd)
xml = '<personnes><personne><nom>A</nom><age>20</age></personne><personne><nom>B</nom><age>30</age></personne></personnes>'
print(f"Valide: {xs.is_valid(xml)}")
print(xs.to_dict(xml))
```

</details>

### Exercice 4 *(moyen)* — Enumération

XSD avec un élément `status` restreint aux valeurs `active`, `pending`, `closed`. Testez avec `xmlschema`.

In [ ]:
# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import xmlschema

xsd = '''<xs:schema xmlns:xs='http://www.w3.org/2001/XMLSchema'>
<xs:element name='status'>
  <xs:simpleType>
    <xs:restriction base='xs:string'>
      <xs:enumeration value='active'/>
      <xs:enumeration value='pending'/>
      <xs:enumeration value='closed'/>
    </xs:restriction>
  </xs:simpleType>
</xs:element>
</xs:schema>'''
xs = xmlschema.XMLSchema(xsd)
for val in ['active', 'pending', 'closed', 'unknown']:
    print(f"'{val}': {xs.is_valid(f'<status>{val}</status>')}")
```

</details>

### Exercice 5 *(difficile)* — XML vers JSON validé

Validez un XML complexe (personnes avec attributs) contre un XSD, puis convertissez en JSON formaté.

In [ ]:
import xmlschema, json

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import xmlschema, json

xsd = '''<xs:schema xmlns:xs='http://www.w3.org/2001/XMLSchema'>
<xs:element name='equipe'>
  <xs:complexType><xs:sequence>
    <xs:element name='membre' maxOccurs='unbounded'>
      <xs:complexType>
        <xs:sequence>
          <xs:element name='nom' type='xs:string'/>
          <xs:element name='role' type='xs:string'/>
        </xs:sequence>
        <xs:attribute name='id' type='xs:integer' use='required'/>
      </xs:complexType>
    </xs:element>
  </xs:sequence></xs:complexType>
</xs:element>
</xs:schema>'''
xml = '<equipe><membre id="1"><nom>Alice</nom><role>dev</role></membre><membre id="2"><nom>Bob</nom><role>ops</role></membre></equipe>'
xs = xmlschema.XMLSchema(xsd)
if xs.is_valid(xml):
    print(json.dumps(xs.to_dict(xml), indent=2, ensure_ascii=False))
```

</details>

### Approfondir : schéma avec import et include

Les schémas XSD complexes sont souvent répartis sur plusieurs fichiers.
`xs:import` permet de référencer un schéma avec un namespace différent,
`xs:include` de fusionner des schémas du même namespace.

In [ ]:
# Exemple conceptuel de structure multi-fichiers
print('''
base_types.xsd:
  <xs:schema targetNamespace="http://example.com/types">
    <xs:simpleType name="monnaie">
      <xs:restriction base="xs:decimal">
        <xs:fractionDigits value="2"/>
      </xs:restriction>
    </xs:simpleType>
  </xs:schema>

commande.xsd:
  <xs:schema>
    <xs:import namespace="http://example.com/types" schemaLocation="base_types.xsd"/>
    <xs:element name="prix" type="types:monnaie"/>
  </xs:schema>
''')

### Exercice : XSD avec choix et séquence

Créez un schéma XSD pour un système de messagerie où :
- Un message peut être de type `texte` OU `image` (utiliser `xs:choice`)
- Un message texte a un `corps` (string) et un `sujet` optionnel
- Un message image a une `url` (anyURI) et une `legende` optionnelle
- Chaque message a un attribut `horodatage` (dateTime) obligatoire

In [ ]:
from lxml import etree

xsd_msg = '''
<xs:schema xmlns:xs="http://www.w3.org/2001/XMLSchema">
  <xs:element name="messages">
    <xs:complexType>
      <xs:sequence>
        <xs:element name="message" maxOccurs="unbounded">
          <xs:complexType>
            <xs:choice>
              <xs:element name="texte">
                <xs:complexType>
                  <xs:sequence>
                    <xs:element name="sujet" type="xs:string" minOccurs="0"/>
                    <xs:element name="corps" type="xs:string"/>
                  </xs:sequence>
                </xs:complexType>
              </xs:element>
              <xs:element name="image">
                <xs:complexType>
                  <xs:sequence>
                    <xs:element name="url" type="xs:anyURI"/>
                    <xs:element name="legende" type="xs:string" minOccurs="0"/>
                  </xs:sequence>
                </xs:complexType>
              </xs:element>
            </xs:choice>
            <xs:attribute name="horodatage" type="xs:dateTime" use="required"/>
          </xs:complexType>
        </xs:element>
      </xs:sequence>
    </xs:complexType>
  </xs:element>
</xs:schema>
'''

schema = etree.XMLSchema(etree.fromstring(xsd_msg.encode()))

xml_test = '''
<messages>
  <message horodatage="2025-06-15T10:30:00">
    <texte><sujet>Hello</sujet><corps>Bonjour le monde</corps></texte>
  </message>
  <message horodatage="2025-06-15T10:31:00">
    <image><url>https://example.com/photo.jpg</url></image>
  </message>
</messages>
'''
print('Valide :', schema.validate(etree.fromstring(xml_test.encode())))

### Exercice récapitulatif : pipeline de validation

Créez une fonction `validate_and_report(xml_path, xsd_path)` qui :
1. Charge le schéma XSD
2. Valide le document XML
3. En cas d'erreur, affiche chaque erreur avec le numéro de ligne
4. Retourne `True`/`False`

In [ ]:
from lxml import etree
from pathlib import Path

def validate_and_report(xml_str: str, xsd_str: str) -> bool:
    """Valide un XML contre un XSD et affiche un rapport."""
    try:
        schema = etree.XMLSchema(etree.fromstring(xsd_str.encode()))
    except etree.XMLSchemaParseError as e:
        print(f'Erreur dans le XSD : {e}')
        return False

    try:
        doc = etree.fromstring(xml_str.encode())
    except etree.XMLSyntaxError as e:
        print(f'XML mal formé : {e}')
        return False

    valid = schema.validate(doc)
    if valid:
        print('Document valide')
    else:
        print(f'{len(schema.error_log)} erreur(s) :')
        for err in schema.error_log:
            print(f'  Ligne {err.line} : {err.message}')
    return valid

# Test
validate_and_report(
    '<catalogue><produit categorie="info"><nom>PC</nom><prix>-10</prix><stock>5</stock></produit></catalogue>',
    xsd_produit  # défini plus haut
)

### Astuce : schéma JSON équivalent

JSON Schema joue le même rôle pour JSON que XSD pour XML.
Comparer les deux formats aide à comprendre les concepts communs
(types, contraintes, structure).

In [ ]:
import json

json_schema = {
    '$schema': 'https://json-schema.org/draft/2020-12/schema',
    'type': 'object',
    'properties': {
        'catalogue': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'nom': {'type': 'string'},
                    'prix': {'type': 'number', 'exclusiveMinimum': 0},
                    'stock': {'type': 'integer', 'minimum': 0},
                    'categorie': {'type': 'string'},
                },
                'required': ['nom', 'prix', 'categorie'],
            }
        }
    }
}
print(json.dumps(json_schema, indent=2))

### Exercice : validation côté client avec xmlschema

Le package `xmlschema` offre une API plus Pythonique que lxml pour la validation.
Il peut aussi convertir XML -> dict et vice-versa.

```python
pip install xmlschema
```

In [ ]:
# Exemple avec xmlschema
try:
    import xmlschema

    xsd = '''
    <xs:schema xmlns:xs="http://www.w3.org/2001/XMLSchema">
      <xs:element name="personne">
        <xs:complexType>
          <xs:sequence>
            <xs:element name="nom" type="xs:string"/>
            <xs:element name="age" type="xs:positiveInteger"/>
          </xs:sequence>
        </xs:complexType>
      </xs:element>
    </xs:schema>
    '''
    schema = xmlschema.XMLSchema(xsd)

    # Validation
    xml_ok = '<personne><nom>Alice</nom><age>30</age></personne>'
    print('Valide :', schema.is_valid(xml_ok))

    # Conversion XML -> dict
    data = schema.to_dict(xml_ok)
    print('Dict :', data)

    # Conversion dict -> XML
    xml_back = schema.encode(data)
    print('XML :', xmlschema.etree_tostring(xml_back))
except ImportError:
    print('xmlschema non installé : pip install xmlschema')

---

## Ressources externes

- [W3C XML Schema Primer](https://www.w3.org/TR/xmlschema-0/)
- [lxml — documentation](https://lxml.de/)
- [xmlschema — documentation](https://xmlschema.readthedocs.io/)